# 04_inference_pipeline

Generated by `notebooks/build_notebooks.py`. Runtime → Change runtime type → GPU, then Run all.

In [ ]:
# @title Setup: GPU check + repo + deps

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
REPO='/content/repo'
!git clone https://github.com/arupa444/trainDronisight.git $REPO 2>/dev/null || (cd $REPO && git pull)
%cd $REPO
!pip -q install uv && uv pip install --system -e .

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available())

In [ ]:
from notebooks.colab_utils import mount_drive, drive_db_zip, ensure_dataset
mount_drive()

In [ ]:
import os; os.environ['DRONISIGHT_DATA'] = '/content/data'  # matches the ensure_dataset() dest below

In [ ]:
ensure_dataset(drive_db_zip('yolo_train_db'), '/content/data', 'yolo_train_db')

In [ ]:
# fresh runtime? pull previously-trained weights back from Drive into runs/
from notebooks.colab_utils import restore_runs_from_drive
print('restored', restore_runs_from_drive(), 'files from Drive')

In [ ]:
# point these at trained weights (now in runs/ after the restore above)
import glob, os
POLE=max(glob.glob('runs/pole/yolo*/weights/best.pt'), key=os.path.getmtime)
COMP=max(glob.glob('runs/components/yolo*/weights/best.pt'), key=os.path.getmtime)
print(POLE, COMP)

In [ ]:
import glob
IMG=sorted(glob.glob('/content/data/yolo_train_db/components/images/test/orig/*.jpg'))[0]
print(IMG)

In [ ]:
!python -m inference.pipeline --image "$IMG" --pole-weights $POLE --comp-weights $COMP --out /content/result.json

In [ ]:
import json; print(json.dumps(json.load(open('/content/result.json')), indent=2))

In [ ]:
# persist the result + crops to Drive
from notebooks.colab_utils import save_runs_to_drive
import shutil, os
os.makedirs('runs/inference', exist_ok=True)
shutil.copy('/content/result.json', 'runs/inference/result.json')
print('saved to', save_runs_to_drive())